
# 08 复杂策略全流程案例：carry + 动量 + regime

> **本章要解决的问题**：一个策略从“有个想法”到“敢不敢放钱”，中间要过几道门？为什么回测 Sharpe 1.3 的策略，最后的结论可能是“先别上”？如果每个环节单独看都没错，为什么合起来会亏？

## 本章导读

前面几本 advanced 教程拆开讲了回测、数据、执行、风险、运维；这一章把它们串成一条**端到端流水线**，主角是一个商品期货的多因子策略：**carry（展期收益）+ 动量 + regime 过滤**。

按“一个人从零开始做策略”的顺序展开：

1. 假设卡与可反驳预测：先写清楚“为什么该赚钱、什么情况下算错”；
2. 数据、universe、换月与 PIT：面板怎么来、为什么不能有未来信息；
3. 标签与因子：5 日/20 日 roll-adjusted 收益、carry、动量、波动；
4. 信号合成与 regime 状态机：什么时候信哪个信号、什么时候关掉；
5. 组合构建：风险预算、波动率目标、约束；
6. 回测与多重检验：成本、滑点、容量、purged CV、DSR、PBO；
7. 风险、归因与 kill switch：回撤、收益拆解、熔断；
8. 模拟盘 → 小资金 → 放大：一张分阶段的 go/no-go 表。

**重要说明**：本章全部数据为**合成教学数据**，参数（成本、ADV、冲击系数）都是教学示例，不代表任何真实市场。方法是可迁移的，数字不能直接拿去下单。

### 学习目标

- **任务 1**：写出一张假设卡，包含经济逻辑、可反驳预测与失败条件。
- **任务 2**：定义 universe 与 PIT 面板，说明换月、存活偏差与数据质量怎么处理。
- **任务 3**：算 5 日/20 日 roll-adjusted 标签与 carry、动量、波动因子，并评估 IC。
- **任务 4**：把因子合成信号，用 regime 状态机做门控，并验证门控是否真的改善风险收益。
- **任务 5**：用风险预算 + 波动率目标 + 约束把信号变成可执行的权重。
- **任务 6**：跑带成本的回测，做 purged CV、DSR、PBO 与容量测算。
- **任务 7**：做风险归因、回撤与 kill switch，并核对归因加总。
- **任务 8**：产出策略卡与分阶段 go/no-go 表，写清下一步修什么。

### 先修

本系列 `03_执行算法与微观结构`、`04_风险模型与收益归因`；主线第 29 章（多重检验）、第 31 章（订单容量）、第 35 章（凯利）、第 39 章（多因子合成）。会 numpy/pandas/scipy。

---

## 在量化研究中的位置

- **谁用**：策略研究员、组合经理、风控、交易/运维。
- **输入**：假设卡、PIT 行情面板、因子定义、成本与容量假设、风控限额。
- **输出**：信号面板、目标权重、带成本回测、多重检验结论、归因表、策略卡、go/no-go 结论。
- **在流程的哪一步**：从“想法”到“账户”的全流程；前面是因子研究，后面接执行与实盘运维。
- **失败模式**：假设不可反驳、PIT 泄漏、标签重叠当独立、regime 门控用未来阈值、成本低估、多重检验缺失、归因对不上、没有 go/no-go 标准。
- **最小工作流**：假设卡 → 数据/PIT → 因子/IC → 信号/regime → 组合 → 回测/多重检验 → 风险/归因 → go/no-go。

---

### 环境准备



In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



---

## 1. 假设卡与可反驳预测

### 1.1 一张假设卡

| 栏目 | 内容 |
|---|---|
| 经济逻辑 | carry 反映期限结构：近月贴水（backwardation）说明现货紧张，多头有展期收益；动量反映供需趋势的持续性 |
| universe | 6 个流动性较好的商品期货品种（教学示例），主力连续合约 |
| 信号 | carry（近远月年化价差）、动量 20 日、波动 20 日 |
| 组合 | 风险预算 + 波动率目标 + regime 门控，做多高 carry/高动量、做空低 carry/低动量 |
| 成本 | 单边 5bp（教学示例），另做 2–20bp 敏感性 |
| 失败条件 | 净 Sharpe < 0.8、DSR < 95%、PBO > 50%、最大回撤 > 15% |

### 1.2 可反驳预测

假设卡里最重要的不是“我为什么对”，而是“**我怎么知道自己错了**”。本章要检验四条：

1. carry 与未来 20 日收益横截面正相关（全样本 IC > 0）；
2. 动量与未来收益正相关；
3. 因子在 risk-on 状态更有效、在 risk-off 状态失效甚至反向；
4. 成本后净 Sharpe 仍显著为正，且能通过 DSR/PBO 门槛。

第 3 条决定了要不要 regime；第 4 条决定能不能上。**如果第 1 条被否定，不代表策略一定死——可能是状态混在一起了，这正是第 4 节要查的。**

---

## 2. 数据、universe、换月与 PIT

### 2.1 面板从哪来

本章用 6 个品种 × 521 个交易日的合成面板（教学数据）。真实世界要做三步：拉取主力合约行情 → 按换月规则拼接连续价格 → 对齐交易日历与结算价。

### 2.2 换月：最容易制造假信号的一步

商品期货有到期日，主力合约会切换。直接拼“主力价格”会在换月日出现跳空，动量、波动、carry 全被污染。处理方式：

- 按持仓量/成交量选主力，设定换月规则（如持仓量连续 2 日反超）；
- 换月时用**价差调整**（roll-adjusted）：新合约价格减去新旧价差，保证收益连续；
- 记录每次换月的日期与价差，事后可以审计。

### 2.3 PIT：每个字段都要有“当时可见”的时间

- carry 用当日收盘可观测的近远月价格；
- 信号在 $t$ 收盘生成，最早 $t+1$ 才能交易；
- 标签（未来 5/20 日收益）只能用于评估，绝不能进特征；
- 复权、修订、退市都要按 as-of 时间对齐。

**合成数据也必须遵守 PIT 纪律**：本章的 `carry_calc`、`mom20` 只由 $t$ 及之前的数据计算，收益用 $t+1$ 起的未来数据，代码里通过错位体现。

---

## 3. 标签与因子

### 3.1 标签：5 日与 20 日

$$\text{fwd5}_t=\frac{P_{t+5}}{P_t}-1,\qquad \text{fwd20}_t=\frac{P_{t+20}}{P_t}-1.$$

两个标签都会**重叠**：相邻两天的 fwd20 共享 19 天收益，IC 序列高度自相关。所以 IC 的显著性不能用普通 $t$ 检验，要分块或 Newey–West 校正；本章只报 IC 均值与 ICIR，不做显著性宣称。

### 3.2 三个因子

- **carry**：$\ln(P_{\text{near}}/P_{\text{far}})\times 365/60$，即近远月年化价差；
- **动量**：$P_{t-1}/P_{t-21}-1$，20 日 roll-adjusted 收益；
- **波动**：20 日已实现波动率，年化。波动不是 alpha，主要给 regime 与仓位用。

截面比较前先做 z-score，避免量纲不同。

### 3.3 代码：面板、标签、因子与 IC



In [2]:

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import norm

rng = np.random.default_rng(8801)
N, T, PAD = 6, 500, 21
names = ["RB", "M", "CU", "TA", "SR", "SC"]          # 教学示例品种代码
dates = pd.bdate_range("2023-01-02", periods=T + PAD)

# 1) 真实 carry：AR(1)，年化（教学数据）
carry_true = np.zeros((T + PAD, N))
carry_true[0] = rng.uniform(-0.06, 0.08, N)
for t in range(1, T + PAD):
    carry_true[t] = 0.98 * carry_true[t - 1] + rng.normal(0, 0.015, N)

# 2) 市场波动状态：三段高波动期（教学设定，模拟 risk-off）
vol_state = np.full(T + PAD, 0.006)
for lo, hi in [(90, 140), (250, 300), (380, 430)]:
    vol_state[lo:hi] = 0.015
mkt = vol_state * rng.normal(0, 1, T + PAD)
beta = np.ones(N)

# 3) 价格与收益分解（市场 / carry / 动量 / 特质）
price = np.zeros((T + PAD, N))
price[0] = np.array([3800.0, 3000.0, 67000.0, 5700.0, 6100.0, 520.0])
ret = np.zeros((T + PAD, N))
mom20 = np.zeros((T + PAD, N))
part = {k: np.zeros((T + PAD, N)) for k in ["市场", "carry", "动量", "特质"]}
for t in range(1, 21):                                # 预热期
    price[t] = price[t - 1] * (1 + rng.normal(0.0002, 0.008, N))
    ret[t] = price[t] / price[t - 1] - 1
    part["特质"][t] = ret[t]
for t in range(21, T + PAD):
    mom20[t] = price[t - 1] / price[t - 21] - 1
    risk_off = vol_state[t - 1] > 0.010
    part["市场"][t] = beta * mkt[t]
    part["carry"][t] = (2.0 if not risk_off else -1.5) * carry_true[t - 1] / 252
    part["动量"][t] = (0.004 if not risk_off else -0.003) * mom20[t]
    part["特质"][t] = rng.normal(0, 0.010, N)
    ret[t] = sum(part[k][t] for k in part)
    price[t] = price[t - 1] * (1 + ret[t])

# 4) 可观测 carry：近远月反推 + 观测噪声
carry_calc = carry_true + rng.normal(0, 0.003, (T + PAD, N))
vol20 = np.full((T + PAD, N), np.nan)
for t in range(20, T + PAD):
    vol20[t] = ret[t - 19:t + 1].std(axis=0, ddof=1) * np.sqrt(252)
vol20 = np.where(np.isnan(vol20), np.nanmedian(vol20), vol20)

fwd5 = np.full((T + PAD, N), np.nan)
fwd20 = np.full((T + PAD, N), np.nan)
for t in range(T + PAD - 20):
    fwd5[t] = price[t + 5] / price[t] - 1
    fwd20[t] = price[t + 20] / price[t] - 1
valid = np.arange(60, T + PAD - 20)

print("面板形状（天数, 品种）：", price.shape, "；合成教学数据，非真实行情")
print("carry 截面标准差均值：", round(float(np.nanstd(carry_calc[valid], axis=1).mean()), 3))
print("年化波动中位数：", np.round(np.nanmedian(vol20[valid], axis=0), 3))

print(f"{'因子':<8}{'标签':<6}{'均值IC':>9}{'ICIR':>8}")
for fname, fac in [("carry", carry_calc), ("动量20", mom20), ("波动20", -vol20)]:
    for lname, lab in [("5日", fwd5), ("20日", fwd20)]:
        ics = np.array([stats.spearmanr(fac[t], lab[t]).statistic for t in valid])
        print(f"{fname:<8}{lname:<6}{ics.mean():>9.3f}{ics.mean() / ics.std(ddof=1):>8.2f}")


面板形状（天数, 品种）： (521, 6) ；合成教学数据，非真实行情
carry 截面标准差均值： 0.058
年化波动中位数： [0.223 0.193 0.207 0.218 0.206 0.201]
因子      标签         均值IC    ICIR


carry   5日       -0.005   -0.01


carry   20日      -0.105   -0.23
动量20    5日        0.108    0.25
动量20    20日       0.139    0.30


波动20    5日       -0.026   -0.06
波动20    20日       0.056    0.11



**读数与应用**：carry 的 5 日 IC 几乎为 0（−0.005），20 日 IC 反而为 −0.105；动量的 20 日 IC 是 +0.139，低波因子 +0.056。如果只看这张表，结论会是“carry 没用、动量还行”。先别下结论：**全样本 IC 把不同状态混在一起了**，而且 20 日标签跨状态边界会污染 IC。第 4 节会按状态拆开看。

**常见错误**：

1. 用主力合约直接拼价格，换月跳空污染动量和波动。
2. 用全样本估计的因子值与未来收益一起算 IC（未来函数）。
3. 把重叠标签当独立样本，IC 的显著性被高估。

---

## 4. 信号合成与 regime 状态机

### 4.1 合成

两个 alpha 因子先 z-score，再线性合成：

$$\text{comp}=0.6\,z(\text{carry})+0.4\,z(\text{mom20}).$$

权重是**预先写进假设卡**的（0.6/0.4，按经济逻辑），不是搜出来的。第 6 节的 DSR/PBO 会检查“如果这组权重是试出来的，结论还成不成立”。

### 4.2 regime 状态机

市场波动用等权市场收益的 20 日已实现波动率，和过去 250 日的中位数比较：

- 进入 risk-off：$\text{vol}>1.30\times\text{median}$，持续至少 3 天；
- 恢复 risk-on：$\text{vol}<1.05\times\text{median}$，持续至少 5 天。

两个不同阈值 + 最短停留就是**滞回**：避免波动率在阈值附近抖动导致频繁开关。状态机只看当日及之前的数据，不用未来信息。

### 4.3 代码：状态机、门控与验证



In [3]:

mkt_ret = ret.mean(axis=1)
mkt_index = np.cumprod(1 + mkt_ret)
vol_mkt = np.full(T + PAD, np.nan)
for t in range(20, T + PAD):
    vol_mkt[t] = mkt_ret[t - 19:t + 1].std(ddof=1) * np.sqrt(252)
vol_mkt = np.where(np.isnan(vol_mkt), np.nanmedian(vol_mkt), vol_mkt)
vol_med = pd.Series(vol_mkt).rolling(250, min_periods=60).median().to_numpy()

state = np.ones(T + PAD, dtype=int)                  # 1=risk-on, 0=risk-off
dwell = 0
for t in range(60, T + PAD):
    if state[t - 1] == 1:
        if vol_mkt[t] > 1.30 * vol_med[t] and dwell >= 3:
            state[t], dwell = 0, 0
        else:
            state[t], dwell = 1, dwell + 1
    else:
        if vol_mkt[t] < 1.05 * vol_med[t] and dwell >= 5:
            state[t], dwell = 1, 0
        else:
            state[t], dwell = 0, dwell + 1


def zscore(a):
    s = np.nanstd(a, axis=1, keepdims=True)
    return (a - np.nanmean(a, axis=1, keepdims=True)) / np.where(s > 0, s, 1)


zc, zm = zscore(carry_calc), zscore(mom20)
comp = 0.6 * zc + 0.4 * zm
sig_gated = comp * state[:, None]
print("regime：risk-on 占比", f"{state[valid].mean():.0%}",
      "，切换", int(np.sum(np.diff(state[valid]) != 0)), "次")
ic_on = [stats.spearmanr(comp[t], ret[t + 1]).statistic for t in valid if state[t] == 1]
ic_off = [stats.spearmanr(comp[t], ret[t + 1]).statistic for t in valid if state[t] == 0]
print("合成信号次日 IC：risk-on", round(float(np.mean(ic_on)), 3),
      "，risk-off", round(float(np.mean(ic_off)), 3))

order = np.argsort(comp, axis=1)
w_ls = np.zeros((T + PAD, N))
for t in range(T + PAD):
    w_ls[t, order[t, -2:]] = 0.25                   # 做多信号最强 2 个
    w_ls[t, order[t, :2]] = -0.25                   # 做空信号最弱 2 个
for tag, ww in [("未门控", w_ls), ("regime门控", w_ls * state[:, None])]:
    pnl = np.nansum(ww[:-1] * ret[1:], axis=1)[60:]
    print(f"{tag}: 年化 {pnl.mean() * 252:.2%}，"
          f"Sharpe {pnl.mean() / pnl.std(ddof=1) * np.sqrt(252):.2f}")


regime：risk-on 占比 61% ，切换 6 次
合成信号次日 IC：risk-on 0.048 ，risk-off -0.005
未门控: 年化 10.70%，Sharpe 1.31
regime门控: 年化 9.68%，Sharpe 1.48



**读数与应用**：risk-on 占 61%、切换 6 次，状态机没有频繁抖。合成信号的次日 IC 在 risk-on 是 +0.048、risk-off 是 −0.005，门控后简单多空 Sharpe 从 1.31 升到 1.48——**门控有用，但提升有限，说明 regime 只是“少亏”，不是“多赚”。** carry 全样本 IC 为负的解释也在这里：risk-off 时因子反向，把全样本平均拉负了。

**常见错误**：

1. 用全样本波动率分位数定阈值（未来函数）。
2. 只用一个阈值，状态一天一变。
3. 把 regime 当方向预测器——它只是开关。

---

## 5. 组合构建：风险预算、波动率目标与约束

### 5.1 从信号到权重

1. **风险预算**：每个品种按 $1/\sigma_i$ 分配风险，再减去截面均值，得到市场中性权重；
2. **波动率目标**：用事前波动（含 0.25 的相关性假设）估计组合波动 $\hat\sigma_p$，杠杆 $L=\min(2,\ 10\%/\hat\sigma_p)$；
3. **约束**：risk-off 时仓位清零；单品种权重上限 40%；实盘再加整数手、最小变动与保证金约束。

### 5.2 为什么要“风险预算 + 波动率目标”

等权会让高波动品种主导组合风险；只做波动率目标又可能让低波动品种权重过大。两者结合：**先按风险分配，再把总风险调到目标水平。**

### 5.3 代码：组合权重



In [4]:

target_vol = 0.10
inv_vol = 1 / np.clip(vol20, 0.05, None)
raw = np.nan_to_num(sig_gated * inv_vol)
raw = raw - np.nanmean(raw, axis=1, keepdims=True)   # 截面去均值 → 市场中性
gross = np.nansum(np.abs(raw), axis=1, keepdims=True)
w_unit = raw / np.where(gross > 0, gross, 1.0)

rho = 0.25                                            # 教学假设的相关性
var_p = np.nansum((w_unit * vol20) ** 2, axis=1)
for i in range(N):
    for j in range(i + 1, N):
        var_p += 2 * rho * np.abs(w_unit[:, i] * w_unit[:, j]) * vol20[:, i] * vol20[:, j]
sigma_p = np.sqrt(np.clip(var_p, 1e-12, None))
lev = np.clip(target_vol / np.where(state == 1, sigma_p, np.inf), 0, 2.0)
W = w_unit * lev[:, None]
W = np.where(state[:, None] == 1, W, 0.0)             # risk-off 清仓
W = np.clip(W, -0.4, 0.4)                             # 单品种上限
turn = np.nansum(np.abs(np.diff(W, axis=0)), axis=1)
print("组合：目标波动", f"{target_vol:.0%}", "，平均杠杆",
      round(float(np.abs(W[60:]).sum(axis=1).mean()), 2),
      "，日均换手", round(float(turn[60:].mean()), 3),
      "，年化换手", round(float(turn[60:].mean() * 252), 1))


组合：目标波动 10% ，平均杠杆 0.53 ，日均换手 0.14 ，年化换手 35.2



**读数与应用**：目标波动 10%，但实际平均杠杆只有 0.53——因为 risk-off 期间仓位为 0，实际波动会低于目标。日均换手 0.14、年化约 35 倍，**换手偏高**：这是后面成本与容量压力的来源。权重被截断再缩放后，gross 可能略偏离 1，实盘要再归一化一次。

**常见错误**：

1. 直接等权，风险被高波动品种主导。
2. 波动率目标用当天已实现波动算当天仓位（未来函数）。
3. 忘记 risk-off 清仓，状态机白做。

---

## 6. 回测、purged CV、DSR/PBO 与容量

### 6.1 回测纪律

- 权重在 $t$ 收盘生成，吃 $t+1$ 的收益；**代码里 `W[:-1] * ret[1:]` 就是这个错位**；
- 成本按换手计：$\text{cost}=5\text{bp}\times\sum|\Delta W|$（教学示例），再做 2–20bp 敏感性；
- 不做“用当天收盘价成交”的假设。

### 6.2 purged CV：给重叠标签留隔离带

把回测区间切成 5 折，每折之间**挖掉 20 天**（purge），避免训练/测试共享重叠标签。这里权重是预先指定的，purged CV 只用于**评估稳定性**，不用于调参；若要调参，必须在训练折内完成，测试折只看一次。

### 6.3 DSR 与 PBO

- **DSR**：试验了 $M=16$ 组 carry/动量权重，按试验间 SR 的离散度算“多重检验后的门槛”$SR_0$，再看策略 SR 超过门槛的概率；
- **PBO**：用 CSCV 把收益切成 8 块、取 4 块做样本内、4 块做样本外，重复 70 次，看“样本内冠军在样本外掉到中位数以下”的比例。

### 6.4 代码：成本、purged CV、多重检验与容量



In [5]:

def backtest(cost_bps):
    return (np.nansum(W[:-1] * ret[1:], axis=1) - turn * cost_bps / 1e4)[60:]


def stats_of(pnl):
    nav = np.cumprod(1 + pnl)
    dd = nav / np.maximum.accumulate(nav) - 1
    return (pnl.mean() * 252, pnl.std(ddof=1) * np.sqrt(252),
            pnl.mean() / pnl.std(ddof=1) * np.sqrt(252), dd.min())


print(f"{'成本bp':>7}{'年化':>9}{'波动':>9}{'Sharpe':>8}{'最大回撤':>11}")
for c in [0, 2, 5, 10, 20]:
    a, v, s, mdd = stats_of(backtest(c))
    print(f"{c:>7}{a:>9.2%}{v:>9.2%}{s:>8.2f}{mdd:>11.2%}")
pnl_net = backtest(5)

fold_edges = np.linspace(0, len(pnl_net), 6).astype(int)
purge, fold_sr = 20, []
for k in range(5):
    lo, hi = fold_edges[k], fold_edges[k + 1]
    seg = pnl_net[lo + purge: hi - purge]
    sd = seg.std(ddof=1)
    fold_sr.append(seg.mean() / sd * np.sqrt(252) if sd > 0 else np.nan)
valid_folds = [x for x in fold_sr if not np.isnan(x)]
print("purged CV 各折 Sharpe:", ["—" if np.isnan(x) else round(float(x), 2) for x in fold_sr])
print("有效折均值", round(float(np.mean(valid_folds)), 2),
      "，标准差", round(float(np.std(valid_folds, ddof=1)), 2))


def ls_returns(signal):
    o = np.argsort(signal, axis=1)
    ww = np.zeros((T + PAD, N))
    for t in range(T + PAD):
        ww[t, o[t, -2:]] = 0.25
        ww[t, o[t, :2]] = -0.25
    return np.nansum(ww[:-1] * ret[1:], axis=1)[60:]


trials = [ls_returns((wc * zc + wm * zm) * state[:, None])
          for wc in [0.3, 0.5, 0.6, 0.8] for wm in [0.2, 0.4, 0.6, 0.8]]
R_trials = np.column_stack(trials)
M, T_obs = R_trials.shape[1], R_trials.shape[0]
sr_trials = R_trials.mean(axis=0) / R_trials.std(axis=0, ddof=1)
skew = float(stats.skew(pnl_net))
kurt = float(stats.kurtosis(pnl_net, fisher=False))
gamma = 0.5772156649
sr0 = np.std(sr_trials, ddof=1) * ((1 - gamma) * norm.ppf(1 - 1 / M)
                                   + gamma * norm.ppf(1 - 1 / (M * np.e)))
sr_hat = pnl_net.mean() / pnl_net.std(ddof=1)
dsr = norm.cdf((sr_hat - sr0) * np.sqrt(T_obs - 1)
               / np.sqrt(1 - skew * sr_hat + (kurt - 1) / 4 * sr_hat ** 2))
print(f"DSR：{M} 组试验，SR0={sr0:.4f}（日频），SR_hat={sr_hat:.4f}，DSR={dsr:.2%}")

S = 8
blocks = np.array_split(np.arange(T_obs), S)
from itertools import combinations
pbo_cnt, splits = 0, 0
for combo in combinations(range(S), S // 2):
    is_idx = np.concatenate([blocks[i] for i in combo])
    oos_idx = np.concatenate([blocks[i] for i in range(S) if i not in combo])
    sr_is = R_trials[is_idx].mean(axis=0) / R_trials[is_idx].std(axis=0, ddof=1)
    best = int(np.argmax(sr_is))
    sr_oos = R_trials[oos_idx].mean(axis=0) / R_trials[oos_idx].std(axis=0, ddof=1)
    pbo_cnt += int(np.mean(sr_oos <= sr_oos[best]) < 0.5)
    splits += 1
pbo = pbo_cnt / splits
print(f"PBO：{splits} 个组合切分，冠军样本外劣于中位的比例 {pbo:.1%}")

gross_avg = float(np.abs(W[60:]).sum(axis=1).mean())
adv_total = 6 * 5e9                                   # 教学假设：合计日成交额 300 亿元
cap_sr = {}
print("容量粗算（冲击系数 0.5、日波动 2%、合计 ADV 300 亿元，教学假设）：")
for cap in [2e7, 1e8, 5e8]:
    notional = turn[60:].mean() * gross_avg * cap
    participation = notional / adv_total
    extra_bps = 0.5 * 0.02 * np.sqrt(max(participation, 1e-8)) * 1e4
    net_c = pnl_net - turn[60:] * extra_bps / 1e4
    cap_sr[cap] = stats_of(net_c)[2]
    print(f"  规模 {cap / 1e8:.1f} 亿：参与率 {participation:.3%}，"
          f"额外冲击 {extra_bps:.2f}bp，净 Sharpe {cap_sr[cap]:.2f}")


   成本bp       年化       波动  Sharpe       最大回撤
      0    4.76%    5.37%    0.89     -5.93%
      2    4.06%    5.36%    0.76     -5.99%
      5    3.00%    5.36%    0.56     -6.24%
     10    1.24%    5.36%    0.23     -6.85%
     20   -2.28%    5.36%   -0.43     -9.35%
purged CV 各折 Sharpe: [1.27, -0.53, '—', -0.4, 0.55]
有效折均值 0.22 ，标准差 0.84
DSR：16 组试验，SR0=0.0484（日频），SR_hat=0.0353，DSR=38.93%
PBO：70 个组合切分，冠军样本外劣于中位的比例 10.0%
容量粗算（冲击系数 0.5、日波动 2%、合计 ADV 300 亿元，教学假设）：
  规模 0.2 亿：参与率 0.005%，额外冲击 0.70bp，净 Sharpe 0.51
  规模 1.0 亿：参与率 0.025%，额外冲击 1.58bp，净 Sharpe 0.46
  规模 5.0 亿：参与率 0.124%，额外冲击 3.52bp，净 Sharpe 0.33



**读数与应用**：毛 Sharpe 0.89、5bp 成本后 0.56、20bp 成本后转负——**成本是这条策略的生死线**。purged CV 有效折均值 0.22、标准差 0.84，折间波动很大，说明收益不稳定。DSR 只有 38.9%，远低于 95% 门槛；PBO 10% 反而很好——说明“试错次数不算多”，但**样本长度和夏普本身不够**支撑这么高的门槛。容量上，1 亿规模净 Sharpe 0.46，5 亿掉到 0.33。

**常见错误**：

1. 用重叠标签的普通 $t$ 检验宣称显著。
2. 用测试集调参，再报 purged CV 的“好结果”。
3. 只报毛 Sharpe，不报成本后；容量用拍脑袋的 ADV。
4. 看到 PBO 低就以为能上，忽略 DSR。

---

## 7. 风险、归因与 kill switch

### 7.1 回撤与熔断

最大回撤 −6.24%、持续 158 天，对 10% 目标波动的策略算温和，但**持续期**很长。kill switch 规则（教学示例）：

- 回撤 < −3%：仓位减半；
- 回撤 < −5%：清仓；
- 回撤恢复到阈值以上：恢复满仓。

熔断把 MDD 压到 −4.79%，但 Sharpe 从 0.56 降到 0.41——**保险是有成本的**，要在“生存”和“收益”之间做选择。

### 7.2 PnL 归因

把日收益拆成市场、carry、动量、特质、成本五项，必须满足：

$$\text{实际 PnL}=\sum\text{分项 PnL}.$$

如果加总对不上，先查权重与收益是否错位、成本是否重复计算、特质项是否漏算。

### 7.3 代码：风险、归因与策略卡



In [6]:

pnl = pnl_net
nav = np.cumprod(1 + pnl)
dd = nav / np.maximum.accumulate(nav) - 1
longest, cur = 0, 0
for x in dd:
    cur = cur + 1 if x < 0 else 0
    longest = max(longest, cur)
print("风险：最大回撤", f"{dd.min():.2%}", "，回撤持续", longest, "天")

exposure = np.ones(len(pnl))
for t in range(1, len(pnl)):
    prev = dd[t - 1]
    exposure[t] = 0.0 if prev < -0.05 else (0.5 if prev < -0.03 else 1.0)
pnl_ks = pnl * exposure
dd_ks = np.cumprod(1 + pnl_ks) / np.maximum.accumulate(np.cumprod(1 + pnl_ks)) - 1
print(f"kill switch：无熔断 MDD {dd.min():.2%} / Sharpe {stats_of(pnl)[2]:.2f}；"
      f"有熔断 MDD {dd_ks.min():.2%} / Sharpe {stats_of(pnl_ks)[2]:.2f}，"
      f"降仓天数 {int((exposure < 1).sum())}")

attrib = {k: np.nansum(W[60:-1] * arr[61:], axis=1) for k, arr in part.items()}
attrib["成本"] = -(turn[60:] * 5 / 1e4)
total = np.nansum(W[60:-1] * ret[61:], axis=1) - turn[60:] * 5 / 1e4
print(f"{'归因项':<8}{'日均bp':>9}")
for k, v in attrib.items():
    print(f"{k:<8}{v.mean() * 1e4:>9.2f}")
resid = abs(sum(v.mean() for v in attrib.values()) - total.mean())
print(f"合计 {sum(v.mean() for v in attrib.values()) * 1e4:.2f}bp，"
      f"实际 {total.mean() * 1e4:.2f}bp，残差 {resid:.1e}")

net_sharpe = stats_of(pnl)[2]
ann_ret, ann_vol, mdd_net = stats_of(pnl)[0], stats_of(pnl)[1], stats_of(pnl)[3]
print("\n策略卡")
for k, v in [("策略", "商品期货 carry+动量+regime"), ("universe", "6 品种（教学示例）"),
             ("净 Sharpe", f"{net_sharpe:.2f}"), ("年化", f"{ann_ret:.2%}"),
             ("波动", f"{ann_vol:.2%}"), ("最大回撤", f"{mdd_net:.2%}"),
             ("日均/年化换手", f"{turn[60:].mean():.2f} / {turn[60:].mean() * 252:.0f}"),
             ("DSR", f"{dsr:.1%}"), ("PBO", f"{pbo:.1%}")]:
    print(f"  {k:<12}{v}")

print("\ngo/no-go 分阶段表")
gates = [
    ("阶段1 模拟盘", [("PBO ≤ 50%", pbo <= 0.5),
                  ("日换手 ≤ 0.5", turn[60:].mean() <= 0.5),
                  ("有 kill switch", True)]),
    ("阶段2 小资金", [("净 Sharpe ≥ 0.8", net_sharpe >= 0.8),
                  ("DSR ≥ 95%", dsr >= 0.95),
                  ("MDD ≥ -15%", mdd_net >= -0.15)]),
    ("阶段3 放大", [("1 亿规模净 Sharpe ≥ 0.3", cap_sr[1e8] >= 0.3),
                  ("滑点/对账校准完成", False)]),
]
for stage, items in gates:
    ok = sum(1 for _, v in items if v)
    detail = "，".join(f"{k} {'✓' if v else '✗'}" for k, v in items)
    print(f"  {stage}：{ok}/{len(items)} 通过（{detail}）")
print("结论：模拟盘可上；小资金暂缓（净 Sharpe/DSR 未达标）；放大不做——先修信号与样本长度。")


风险：最大回撤 -6.24% ，回撤持续 158 天
kill switch：无熔断 MDD -6.24% / Sharpe 0.56；有熔断 MDD -4.79% / Sharpe 0.41，降仓天数 143
归因项          日均bp
市场           0.01
carry        2.13
动量           0.67
特质          -0.92
成本          -0.70
合计 1.19bp，实际 1.19bp，残差 2.7e-20

策略卡
  策略          商品期货 carry+动量+regime
  universe    6 品种（教学示例）
  净 Sharpe    0.56
  年化          3.00%
  波动          5.36%
  最大回撤        -6.24%
  日均/年化换手     0.14 / 35
  DSR         38.9%
  PBO         10.0%

go/no-go 分阶段表
  阶段1 模拟盘：3/3 通过（PBO ≤ 50% ✓，日换手 ≤ 0.5 ✓，有 kill switch ✓）
  阶段2 小资金：1/3 通过（净 Sharpe ≥ 0.8 ✗，DSR ≥ 95% ✗，MDD ≥ -15% ✓）
  阶段3 放大：1/2 通过（1 亿规模净 Sharpe ≥ 0.3 ✓，滑点/对账校准完成 ✗）
结论：模拟盘可上；小资金暂缓（净 Sharpe/DSR 未达标）；放大不做——先修信号与样本长度。



**读数与应用**：归因表显示 carry 贡献 2.13bp/天、动量 0.67bp/天，市场项约 0（已经市场中性），特质项 −0.92bp/天、成本 −0.70bp/天，合计与实际完全一致（残差 $10^{-20}$ 量级）。**归因的结论很直接：收益主要来自 carry，动量贡献有限，特质为负说明选品有运气成分。** 策略卡把关键指标压成一行行，方便拿去评审。

**常见错误**：

1. 只报总收益，不拆归因，不知道钱从哪来。
2. 熔断规则写在文档里，没写进代码。
3. 归因加总不核对，差 0.5bp 也当“差不多”。

---

## 8. 模拟盘 → 小资金 → 放大：go/no-go

### 8.1 三阶段门槛

| 阶段 | 目标 | 关键条件 | 本案例结果 |
|---|---|---|---|
| 模拟盘 | 验证流程与信号 | 数据/对账通过、PBO ≤ 50%、换手可执行 | 3/3 通过 |
| 小资金 | 验证成本与偏差 | 净 Sharpe ≥ 0.8、DSR ≥ 95%、MDD ≥ −15%、实际滑点 ≈ 模型 | 1/3 不通过 |
| 放大 | 验证容量与运维 | 容量达标、滑点/对账校准、监控演练 | 1/2（容量过，校准未做） |

**关键**：不要一次性“从回测跳到实盘”。模拟盘验证“流程对不对”，小资金验证“成本真不真”，放大验证“容量撑不撑得住”。

### 8.2 本案例的结论

这一轮 go/no-go 的结论是：**模拟盘可以上；小资金暂缓；放大不做。** 原因写清楚：净 Sharpe 0.56 < 0.8、DSR 38.9% < 95%，样本只有 500 天、策略容量对成本敏感。**敢于说“不”，是 go/no-go 清单最大的价值。**

### 8.3 失败模式与复盘模板

- **回测过拟合**：权重、阈值、状态机都可能是“看着历史挑的”；
- **成本低估**：5bp 只是教学示例，真实冲击随规模上升；
- **换月处理错**：roll-adjusted 没做对，动量与波动全是假信号；
- **regime 滞后**：波动率上升后状态才切换，尾部照样吃满；
- **容量误判**：小资金没事，规模上去后冲击吃掉收益；
- **执行与运维**：滑点、拒单、断线、漏记成交。

复盘模板四行：**结论（继续/暂停/放弃）、证据（指标 + 对账）、剩余风险、下一步动作与负责人。**

---

## 本章小结

### 一张表：八道门

| 环节 | 产物 | 通过标准（教学示例） |
|---|---|---|
| 假设卡 | 经济逻辑 + 可反驳预测 | 预测可检验 |
| 数据/PIT | 面板 + 换月记录 | 无未来信息 |
| 因子/IC | IC、ICIR | 方向符合假设 |
| regime | 状态序列 | 有滞回、切换不频繁 |
| 组合 | 目标权重 | 风险预算 + 约束 |
| 回测/多重检验 | 净 Sharpe、DSR、PBO | 净 Sharpe ≥ 0.8、DSR ≥ 95%、PBO ≤ 50% |
| 风险/归因 | 回撤、归因表 | 加总对得上、熔断可执行 |
| go/no-go | 策略卡 + 分阶段表 | 每阶段条件明确 |

### 八句话

1. 假设卡先写“什么算错”，再谈“为什么对”。
2. 换月不做 roll-adjusted，动量和波动都是假的。
3. PIT 纪律在合成数据上也要守。
4. 重叠标签必须 purge，否则 CV 会骗你。
5. regime 是开关，不是方向预测器。
6. 成本 5bp 能吃掉半条策略的 Sharpe。
7. PBO 低不等于能上，DSR 才是多重检验后的门槛。
8. go/no-go 的价值在于敢于说“不”。

---

## 本章习题

**1.** 一张假设卡至少要写清哪几项？为什么“可反驳预测”比“为什么赚钱”更重要？

**2.** 主力合约直接拼接会导致什么问题？roll-adjusted 的作用是什么？

**3.** 5 日/20 日标签重叠会带来什么统计问题？purged CV 的 purge 是为了解决什么？

**4.** 全样本 carry IC 为负，为什么不能据此直接否定 carry 因子？下一步应该查什么？

**5.** regime 状态机为什么要用两个阈值 + 最短停留？只用单阈值会发生什么？

**6.** 策略毛 Sharpe 0.89、5bp 成本后 0.56、20bp 成本后转负，说明什么？如何用容量表找到规模上限？

**7.** DSR 38.9%、PBO 10%，这两个数怎么一起读？能不能上小资金？

**8.** 归因表里 carry 2.13bp、动量 0.67bp、特质 −0.92bp、成本 −0.70bp，合计应为多少？如果对不上，先查什么？

<details><summary>参考答案</summary>

**1.** 经济逻辑、universe、信号、组合规则、成本假设、失败条件。因为“为什么赚钱”可以事后编故事，可反驳预测才能让数据判对错。

**2.** 换月日价格跳空，动量和波动被污染，carry 也可能失真。roll-adjusted 用价差调整让连续价格收益平滑，避免假信号。

**3.** 重叠标签使 IC 序列自相关，普通 $t$ 检验高估显著性；purged CV 在训练/测试折之间挖掉一段时间，避免标签共享。

**4.** 因为不同 regime 下因子方向可能相反，全样本平均会把正负抵消甚至拉负；下一步按状态拆开算条件 IC，并检查标签是否跨状态边界。

**5.** 两个阈值形成滞回，最短停留防止抖动；单阈值会让状态在阈值附近频繁切换，换手和成本爆炸。

**6.** 策略对成本高度敏感，容量有限。用成本敏感性表找到净 Sharpe 跌破可接受线的成本，再用冲击模型反推规模；本案例 1 亿规模净 Sharpe 0.46、5 亿 0.33。

**7.** PBO 10% 说明“试的参数组不多、冠军在样本外没有严重翻转”，但 DSR 38.9% 说明扣掉多重检验门槛后不显著。小资金仍不能上，至少要先把净 Sharpe 与样本长度提上来。

**8.** 合计 2.13 + 0.67 − 0.92 − 0.70 = 1.18bp/天，应与实际 PnL 一致；对不上先查权重与收益错位、成本重复计算、特质项是否漏算。

</details>

---

## 参考资料

- López de Prado, M. *Advances in Financial Machine Learning*（purged CV、PBO、DSR）。
- Grinold, R. C. & Kahn, R. N. *Active Portfolio Management*（信息比与信号合成）。
- Chan, E. P. *Algorithmic Trading: Winning Strategies and Their Rationale*（均值回归、动量与成本）。
- 本系列 `03_执行算法与微观结构`、`04_风险模型与收益归因`；主线第 29、31、35、39 章。
